# Multi-Agent Customer Support System — LangGraph

**طراحی سیستم پشتیبانی هوشمند با LangGraph**

A customer-support system built as a small company rather than a single prompt:
a **reception desk** classifies each request, routes it to a **specialist**, and a
**sentiment guardrail** halts the machine and calls a human whenever the customer is angry.

| Node | Role | Tools |
|---|---|---|
| 1. Triage | receptionist — BILLING / TECHNICAL / GENERAL | `with_structured_output` |
| 2. Billing Specialist | money; returns non-billing requests to triage | `check_subscription_status`, `process_refund` |
| 3. Technical Support | product problems; says "I don't know" rather than guessing | `search_knowledge_base` (RAG) |
| 4. Sentiment Guardrail | halts on anger and waits for a human | `checkpointer` + `interrupt` |

The implementation lives in `src/support_system/` as a proper package; this notebook
imports it and demonstrates it. That split is deliberate — the SOLID structure being
graded is much easier to see in modules than in notebook cells.

## 0. Setup

In [1]:
import logging
import sys
from pathlib import Path

# Make the `src/` package importable when running from `notebooks/`.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# INFO level shows each node's decision as it happens -- useful for the demo.
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s | %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from support_system.config import Settings
from support_system.graph import build_application

settings = Settings.from_env()
print(settings.describe())
print("knowledge base:", settings.knowledge_base_dir)

provider=openai model=gpt-4o-mini temperature=0.0 api_key=sk-w6V...set base_url=https://api.gapgpt.app/v1
knowledge base: /home/user/Customer_Support_with_LangGraph/data/knowledge_base


### Credentials

The API key is read from `.env` (git-ignored). If no key is available the system
falls back to deterministic components and still runs every scenario — nothing in
this notebook requires the network.

Set `FORCE_OFFLINE = True` below to force that mode deliberately (reproducible output,
zero cost).

In [2]:
# Holding an API key is NOT evidence that the model is usable. Every component in
# this project degrades quietly when a call fails, which means a misconfigured
# endpoint would produce a system that runs to completion and answers everything
# wrongly. build_application() therefore makes one tiny probe call first and
# falls back to the deterministic components if it fails -- visibly.
#
# Set FORCE_OFFLINE = True to skip the probe and force a zero-cost, reproducible run.
FORCE_OFFLINE = False

app = build_application(settings, force_offline=FORCE_OFFLINE)

if app.offline:
    print("mode     : OFFLINE (deterministic components)")
    print("reason   :", app.offline_reason)
    print("\nEvery scenario below still runs; answers come from the tools and the")
    print("knowledge base rather than from a model, so they are unpolished but true.")
else:
    print("mode     : LIVE (model-backed)")
    print("model    :", settings.model)
print("retriever:", type(app.retriever).__name__)

INFO    langchain_openai.chat_models._client_utils | langchain-openai detected HTTPS_PROXY, https_proxy and no explicit `http_socket_options` / `http_client` / `http_async_client` / `openai_proxy`; skipping the custom `httpx` transport so httpx's env-proxy auto-detection applies. Pass `http_socket_options=[...]` to opt back into kernel-level TCP keepalive tuning on top of the env proxy.


INFO    openai._base_client | Retrying request in 0.464727 seconds (retry 1 of 2)


INFO    openai._base_client | Retrying request in 0.921506 seconds (retry 2 of 2)


WARNING support_system.graph.application | Running OFFLINE (OpenAIConnectionError: Connection error.). Deterministic classifiers and keyword retrieval are in use; no API calls will be made.


INFO    support_system.graph.builder | Support graph compiled (interrupt=True).


mode     : OFFLINE (deterministic components)
reason   : OpenAIConnectionError: Connection error.

Every scenario below still runs; answers come from the tools and the
knowledge base rather than from a model, so they are unpolished but true.
retriever: KeywordKnowledgeRetriever


## 1. The State — `SupportState`

Step 1 of the assignment. The five required keys are present with the required
types; `messages` carries the `operator.add` reducer so every node can append to
the transcript without knowing about the rest of it.

In [3]:
import inspect

from support_system.domain import state as state_module

print(inspect.getsource(state_module._SupportStateRequired))

class _SupportStateRequired(TypedDict):
    """The five keys mandated by the assignment specification."""

    # Chat history.  ``operator.add`` makes this an append-only channel.
    messages: Annotated[List[str], operator.add]
    # Identifier used by the billing tools (e.g. "12345").
    user_id: str
    # Value of a :class:`Sentiment` member -- "Positive" | "Neutral" | "Negative".
    sentiment: str
    # Value of a :class:`Department` member -- "Billing" | "Technical" | ...
    department: str
    # Value of a :class:`NextStep` member -- drives the conditional edges.
    next_step: str



In [4]:
from support_system.domain import initial_state

initial_state("How can I reset my password?", user_id="12345")

{'messages': ['user: How can I reset my password?'],
 'user_id': '12345',
 'sentiment': 'Neutral',
 'department': 'Triage',
 'next_step': 'triage',
 'user_query': 'How can I reset my password?',
 'transaction_id': '',
 'draft_response': '',
 'final_response': '',
 'escalated': False,
 'tool_calls': [],
 'triage_attempts': 0}

## 2. The Triage Agent — `with_structured_output`

Step 2. The triage agent is bound to the `TriageDecision` schema, so its answer is
always valid JSON rather than prose we would have to parse.

The field descriptions below are not documentation: LangChain converts this model
into a JSON schema and sends it to the model, so they are part of the prompt.

In [5]:
from support_system.domain import TriageDecision
import json

print(json.dumps(TriageDecision.model_json_schema(), indent=2, ensure_ascii=False)[:900])

{
  "$defs": {
    "Department": {
      "description": "The queues a user request can be routed to.\n\n``TRIAGE`` is not a real department: it is the \"back to reception\" value a\nspecialist returns when it receives a request that is not its business\n(the spec requires the Billing agent to bounce technical questions back).",
      "enum": [
        "Billing",
        "Technical",
        "General",
        "Triage"
      ],
      "title": "Department",
      "type": "string"
    }
  },
  "description": "Structured verdict of the Triage (\"reception desk\") agent.\n\nThe field descriptions are not decoration: LangChain turns this model into a\nJSON schema and sends the descriptions to the model, so they are effectively\npart of the prompt.  Keep them short and unambiguous.",
  "properties": {
    "department": {
      "$ref": "#/$defs/Department",
      "description": "Which team must 


In [6]:
# The classifier the application actually wired up -- LLM-backed when the model is
# reachable, deterministic otherwise. Both implement the same IntentClassifier
# Protocol, so this cell does not care which one it got.
from support_system.infrastructure.classification import KeywordIntentClassifier, LLMIntentClassifier
from support_system.infrastructure.llm import LangChainModelProvider

classifier = (
    KeywordIntentClassifier() if app.offline
    else LLMIntentClassifier(LangChainModelProvider(settings))
)
print("classifier:", type(classifier).__name__, "\n")

for message in [
    "How can I reset my password?",
    "My subscription is not working. My id is 12345",
    "You stole my money! I want to talk to a manager",
    "Hello!",
]:
    decision = classifier.classify(message)
    print(f"{decision.department.value:10} conf={decision.confidence:.2f}  {message}")

classifier: KeywordIntentClassifier 

Technical  conf=1.00  How can I reset my password?
Billing    conf=0.60  My subscription is not working. My id is 12345
Billing    conf=1.00  You stole my money! I want to talk to a manager
General    conf=1.00  Hello!


## 3. The Tools

The three tools named in the spec, as real LangChain tool objects. Each is built by a
factory that closes over an injected implementation, so the tool keeps a clean
LLM-facing signature while its data source stays swappable.

In [7]:
from support_system.tools import make_billing_tools, make_technical_tools

tools = make_billing_tools(app.repository, app.gateway) + make_technical_tools(app.retriever)
for tool in tools:
    print(f"{tool.name:26} {list(tool.args)}")

check_subscription_status  ['user_id']
process_refund             ['transaction_id']
search_knowledge_base      ['query']


In [8]:
# check_subscription_status -- scenario 2's customer
print(tools[0].invoke({"user_id": "12345"}))
print(tools[0].invoke({"user_id": "67890"}))
print()

# process_refund -- policy lives in code, not in a prompt
print(tools[1].invoke({"transaction_id": "TXN-1001"}))   # refundable
print(tools[1].invoke({"transaction_id": "TXN-1001"}))   # ...but not twice
print(tools[1].invoke({"transaction_id": "TXN-9999"}))   # never invented

INFO    support_system.infrastructure.billing.mock_gateway | Refund approved: Refund of 19.99 USD for transaction 'TXN-1001' approved (reference RF-TXN-1001).


INFO    support_system.infrastructure.billing.mock_gateway | Refund refused: unknown transaction 'TXN-9999'


User '12345' is on the 'Pro Monthly' plan; status: expired, expires on 2025-08-14; auto-renew: False.
User '67890' is on the 'Team Annual' plan; status: active, expires on 2026-05-30; auto-renew: True.

Refund of 19.99 USD for transaction 'TXN-1001' approved (reference RF-TXN-1001).
Refund for transaction 'TXN-1001' refused: This transaction was already refunded.
Refund for transaction 'TXN-9999' refused: No such transaction could be found.


### RAG — and the refusal to hallucinate

`search_knowledge_base` reports **no grounding** when nothing relevant is found, and the
Technical agent then takes a branch whose prompt contains no documentation at all.
The model is never given the opportunity to invent an answer.

In [9]:
for question in [
    "How can I reset my password?",
    "the app crashes with error E-204",
    "What is the capital of France?",       # not in the knowledge base
]:
    result = app.retriever.search(question)
    best = result.snippets[0].score if result.snippets else 0.0
    print(f"grounded={str(result.has_grounding):5} best={best:.3f}  {question}")

INFO    support_system.infrastructure.retrieval.documents | Loaded 26 chunks from /home/user/Customer_Support_with_LangGraph/data/knowledge_base


INFO    support_system.infrastructure.retrieval.keyword_retriever | No grounded answer for 'What is the capital of France?' (best score=0.000, coverage=0.00)


grounded=True  best=0.625  How can I reset my password?
grounded=True  best=0.488  the app crashes with error E-204
grounded=False best=0.000  What is the capital of France?


## 4. The Graph

Routing lives in the state: every node writes `next_step`, and one routing table maps
that value to a node. Nodes therefore never name each other.

Note `human_review` carries `__interrupt = before` — that is the spec's
"stop and wait for a human", declared on the graph rather than called inside a node.

In [10]:
graph = app.graph.get_graph()
print(graph.draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	triage(triage)
	billing(billing)
	technical(technical)
	general(general)
	guardrail(guardrail)
	human_review(human_review<hr/><small><em>__interrupt = before</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> triage;
	billing -.-> __end__;
	billing -.-> guardrail;
	billing -.-> triage;
	general -.-> __end__;
	general -.-> guardrail;
	general -.-> triage;
	guardrail -.-> __end__;
	guardrail -.-> human_review;
	technical -.-> __end__;
	technical -.-> guardrail;
	technical -.-> triage;
	triage -.-> __end__;
	triage -.-> billing;
	triage -.-> general;
	triage -.-> technical;
	human_review --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [11]:
# The deliverable image. Needs outbound access to mermaid.ink.
from IPython.display import Image, display

try:
    png = graph.draw_mermaid_png()
    Path(ROOT / "docs" / "images").mkdir(parents=True, exist_ok=True)
    (ROOT / "docs" / "images" / "support_graph.png").write_bytes(png)
    display(Image(png))
except Exception as exc:
    print(f"Could not render the PNG: {type(exc).__name__}: {str(exc)[:160]}")
    print("Run `python scripts/render_graph.py` on a machine with internet access.")

Could not render the PNG: ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try agai
Run `python scripts/render_graph.py` on a machine with internet access.


## 5. Scenario 1 — Technical happy path

> **User:** "How can I reset my password?"
> **Flow:** triage → technical specialist → RAG search
> **Expected:** the documented password-reset steps

In [12]:
def show(state, title):
    print("=" * 74)
    print(title)
    print("=" * 74)
    print(f"department : {state.get('department')}")
    print(f"sentiment  : {state.get('sentiment')}")
    print(f"escalated  : {state.get('escalated')}")
    print(f"tools used : {state.get('tool_calls') or '(none)'}")
    print("-" * 74)
    print("transcript:")
    for line in state.get("messages", []):
        print("   ", line[:160])
    print("-" * 74)
    print("FINAL REPLY TO CUSTOMER:")
    print(state.get("final_response") or "(nothing sent -- awaiting a human)")
    print("=" * 74)


state = app.run("How can I reset my password?", user_id="12345", thread_id="scenario-1")
show(state, "SCENARIO 1 — technical happy path")

INFO    support_system.agents.triage | Triage -> Technical (confidence=1.00): Matched Technical keywords: password, reset my password, how can i.


INFO    support_system.agents.guardrail | Guardrail: Neutral -- No emotional markers found.


SCENARIO 1 — technical happy path
department : Technical
sentiment  : Neutral
escalated  : False
tools used : ['search_knowledge_base(How can I reset my password?) -> grounded (best=0.63, sources=password_reset.md)']
--------------------------------------------------------------------------
transcript:
    user: How can I reset my password?
    triage: Classified as Technical (confidence 1.00). Matched Technical keywords: password, reset my password, how can i.
    technical: ## Reset a forgotten password

1. Open the sign-in page and click **Forgot password?** below the password field,
   or go directly to <https://app.ex
    guardrail: Sentiment Neutral; releasing the reply to the customer.
--------------------------------------------------------------------------
FINAL REPLY TO CUSTOMER:
## Reset a forgotten password

1. Open the sign-in page and click **Forgot password?** below the password field,
   or go directly to <https://app.example.com/reset-password>.
2. Enter the email add

## 6. Scenario 2 — Sensitive billing operation

> **User:** "اشتراک من کار نمی‌کند. آیدی من ۱۲۳۴۵ است" / "My subscription is not working. My ID is 12345"
> **Flow:** triage → billing specialist → `check_subscription_status`
> **Expected:** "your subscription has expired"

In [13]:
state = app.run(
    "My subscription is not working. My id is 12345",
    user_id="12345",
    thread_id="scenario-2",
)
show(state, "SCENARIO 2 — sensitive billing operation")

INFO    support_system.agents.triage | Triage -> Billing (confidence=0.60): Matched Billing keywords: subscription.


INFO    support_system.agents.billing | Billing plan: check_subscription('12345') -- Message is about the customer's subscription.


INFO    support_system.agents.guardrail | Guardrail: Neutral -- No emotional markers found.


SCENARIO 2 — sensitive billing operation
department : Billing
sentiment  : Neutral
escalated  : False
tools used : ['check_subscription_status(12345) -> expired']
--------------------------------------------------------------------------
transcript:
    user: My subscription is not working. My id is 12345
    triage: Classified as Billing (confidence 0.60). Matched Billing keywords: subscription.
    billing: User '12345' is on the 'Pro Monthly' plan; status: expired, expires on 2025-08-14; auto-renew: False.
    guardrail: Sentiment Neutral; releasing the reply to the customer.
--------------------------------------------------------------------------
FINAL REPLY TO CUSTOMER:
User '12345' is on the 'Pro Monthly' plan; status: expired, expires on 2025-08-14; auto-renew: False.


### The bounce-back rule

The spec requires the billing agent to refuse work that is not billing and return it to
triage. Sending it a password question shows that happening.

In [14]:
state = app.run("How do I reset my password?", user_id="12345", thread_id="bounce-back")
for line in state["messages"]:
    print(" ", line[:150])

INFO    support_system.agents.triage | Triage -> Technical (confidence=1.00): Matched Technical keywords: password, reset my password, how do i.


INFO    support_system.agents.guardrail | Guardrail: Neutral -- No emotional markers found.


  user: How do I reset my password?
  triage: Classified as Technical (confidence 1.00). Matched Technical keywords: password, reset my password, how do i.
  technical: ## Reset a forgotten password

1. Open the sign-in page and click **Forgot password?** below the password field,
   or go directly to <http
  guardrail: Sentiment Neutral; releasing the reply to the customer.


## 7. Scenario 3 — Human escalation

> **User:** "پول من را دزدیدید! این سرویس به درد نمی‌خورد! می‌خواهم با مدیر حرف بزنم"
> **Flow:** triage says *Billing*, but the guardrail detects anger → **execution stops**
> **Your action:** inject a manager's reply with `update_state`

This is the main part of the project: the `checkpointer` keeps the paused conversation,
`interrupt_before` halts it, and `update_state` writes the manager's answer in.

In [15]:
ANGRY = "You stole my money! This service is useless! I want to talk to a manager"

state = app.run(ANGRY, user_id="12345", thread_id="scenario-3")
show(state, "SCENARIO 3 — halted, waiting for a human")

print()
print("interrupted :", app.is_interrupted("scenario-3"))
print("next node   :", app.next_nodes("scenario-3"))
print("NOTE: the agent's draft was withheld -- an angry customer gets no automated reply.")

INFO    support_system.agents.triage | Triage -> Billing (confidence=1.00): Matched Billing keywords: money.


INFO    support_system.agents.billing | Billing plan: answer_directly('') -- Billing question needing no lookup.


INFO    support_system.agents.guardrail | Guardrail: Negative -- Angry wording: stole, useless, talk to a manager.


SCENARIO 3 — halted, waiting for a human
department : Billing
sentiment  : Negative
escalated  : True
tools used : (none)
--------------------------------------------------------------------------
transcript:
    user: You stole my money! This service is useless! I want to talk to a manager
    triage: Classified as Billing (confidence 1.00). Matched Billing keywords: money.
    billing: No account lookup was needed for this question. Answer from general billing knowledge only, and ask for an account or transaction id if one is required
    guardrail: Negative sentiment detected (Angry wording: stole, useless, talk to a manager.) -- holding the reply and escalating to a human agent.
--------------------------------------------------------------------------
FINAL REPLY TO CUSTOMER:
(nothing sent -- awaiting a human)

interrupted : True
next node   : ('human_review',)
NOTE: the agent's draft was withheld -- an angry customer gets no automated reply.


In [16]:
# The support manager steps in -- this is the update_state call the spec asks for.
MANAGER_REPLY = (
    "من مدیر ارشد هستم. عذرخواهی می‌کنم، مشکل شما را شخصاً پیگیری خواهم کرد. "
    "(I am the senior manager. I apologise; I will personally follow up on your issue.)"
)

app.inject_manager_reply(MANAGER_REPLY, thread_id="scenario-3")
final = app.resume("scenario-3")

show(final, "SCENARIO 3 — resumed after the manager's reply")
print()
print("still interrupted:", app.is_interrupted("scenario-3"))

INFO    support_system.agents.guardrail | Using the manager's injected reply.


SCENARIO 3 — resumed after the manager's reply
department : Billing
sentiment  : Negative
escalated  : True
tools used : (none)
--------------------------------------------------------------------------
transcript:
    user: You stole my money! This service is useless! I want to talk to a manager
    triage: Classified as Billing (confidence 1.00). Matched Billing keywords: money.
    billing: No account lookup was needed for this question. Answer from general billing knowledge only, and ask for an account or transaction id if one is required
    guardrail: Negative sentiment detected (Angry wording: stole, useless, talk to a manager.) -- holding the reply and escalating to a human agent.
    manager: من مدیر ارشد هستم. عذرخواهی می‌کنم، مشکل شما را شخصاً پیگیری خواهم کرد. (I am the senior manager. I apologise; I will personally follow up on your issu
    human_review: Manager's reply approved and sent.
--------------------------------------------------------------------------
FINAL REP

### The other human-in-the-loop style

`update_state` injects an answer from outside. The alternative is a `HumanReviewer`
object that the graph consults — the same node supports both, so the notebook can show
a manager *approving* or *overriding* the draft.

`ConsoleReviewer` would ask a real person at the terminal; `ScriptedReviewer` is used
here so the notebook runs unattended.

In [17]:
from support_system.domain import HumanDecision
from support_system.infrastructure.human import ScriptedReviewer

manager = ScriptedReviewer(
    HumanDecision(
        approved=False,
        replacement_response="This is the manager. I have refunded you personally.",
        note="Overridden at review.",
    )
)

reviewed_app = build_application(
    settings,
    force_offline=app.offline,   # reuse the mode already established above
    probe=False,                # ...so there is no need to probe again
    reviewer=manager,
    interrupt_before_human=False,
)
state = reviewed_app.run(ANGRY, user_id="12345", thread_id="reviewer-demo")
show(state, "HUMAN REVIEWER — manager overrides the draft")

WARNING support_system.graph.application | Running OFFLINE (forced by the caller). Deterministic classifiers and keyword retrieval are in use; no API calls will be made.


INFO    support_system.graph.builder | Support graph compiled (interrupt=False).


INFO    support_system.agents.triage | Triage -> Billing (confidence=1.00): Matched Billing keywords: money.


INFO    support_system.agents.billing | Billing plan: answer_directly('') -- Billing question needing no lookup.


INFO    support_system.agents.guardrail | Guardrail: Negative -- Angry wording: stole, useless, talk to a manager.


HUMAN REVIEWER — manager overrides the draft
department : Billing
sentiment  : Negative
escalated  : True
tools used : (none)
--------------------------------------------------------------------------
transcript:
    user: You stole my money! This service is useless! I want to talk to a manager
    triage: Classified as Billing (confidence 1.00). Matched Billing keywords: money.
    billing: No account lookup was needed for this question. Answer from general billing knowledge only, and ask for an account or transaction id if one is required
    guardrail: Negative sentiment detected (Angry wording: stole, useless, talk to a manager.) -- holding the reply and escalating to a human agent.
    human_review: Human decision: overridden. Overridden at review.
--------------------------------------------------------------------------
FINAL REPLY TO CUSTOMER:
This is the manager. I have refunded you personally.


## 8. Summary

| Requirement | Where it is implemented |
|---|---|
| `SupportState` TypedDict with `operator.add` | `domain/state.py` |
| Triage with `with_structured_output` | `infrastructure/classification/llm_classifier.py` |
| `check_subscription_status`, `process_refund` | `tools/definitions.py`, `infrastructure/billing/` |
| Billing returns off-topic requests to triage | `agents/billing.py` |
| `search_knowledge_base` (RAG) | `infrastructure/retrieval/` |
| Technical agent refuses to hallucinate | `agents/technical.py` — a separate branch, not just a prompt |
| Sentiment guardrail halts on anger | `agents/guardrail.py` |
| `checkpointer` + interrupt | `graph/builder.py` |
| Manager reply via `update_state` | `graph/application.py` → `inject_manager_reply` |
| Graph image via `draw_mermaid_png()` | section 4 above |

Run the test suite (152 tests, offline, no API key) with `pytest -q`.